# Executive Overview: Credit Card Fraud Detection Pipeline

Brian Leung

Jupyter Notebook on GitHub: https://github.com/aybara134/cdl20-capstone/blob/main/jupyter/anomaly_detection_creditcard.ipynb

---

Financial institutions process millions of transactions every day. A very small fraction of these may be fraudulent, but identifying them quickly is critical to reducing financial losses and reputational damage.

This notebook demonstrates how an Azure Machine Learning pipeline can automatically analyze transaction data and identify suspicious activity using machine learning. The goal is not to replace human investigators, but to prioritize the highest-risk transactions for review, allowing fraud teams to focus their efforts where they matter most.

The workflow consists of several steps:

1. Data Preparation - Historical data is cleaned and structured so that machine learning algorithms can analyze it effectively.
2. Model Training - Historical transactions labeled as fraudulent or legitimate are used to train a predictive model.
3. Model Evaluation - The model is tested to measure how accurately it detects fraud.
4. Model Registration - Once validated, the model is stored in Azure ML for governance and deployment readiness.
5. Deployment Preparation - The model can then be deployed as a service to analyze new transactions in real time.

### End-to-End Pipeline

```
+------------------------+
|  Historical Data       |
+-----------+------------+
            |
            v
+------------------------+
| Data Preparation       |
+-----------+------------+
            |
            v
+------------------------+
| Model Training         |
+-----------+------------+
            |
            v
+------------------------+
| Model Evaluation       |
+-----------+------------+
            |
            v
+------------------------+
| Model Registration     |
+-----------+------------+
            |
            v
+------------------------+
| Deployment             |
+------------------------+
```

### Azure Machine Learning Components

| Azure ML Component | Role in the Fraud Detection Process                                                                         |
| -------------------| ------------------------------------------------------------------------------------------------------------|
| Workspace          | A hub to organize, track, and share all the artifacts, resources, and results of ML projects.               |
| Compute Cluster    | Managed, scalable infrastructure used for running training jobs and batch inference workloads in the cloud. |
| Data Asset         | Structured representation of historical credit card transactions used for training and testing the model.   |
| Experiment         | Collection of runs used to validate the fraud prediction model and track performance metrics.               |
| Model Registry     | Storage for the trained fraud predication model for version control and deployment.                         |
| Endpoint           | Service that allows applications to send requests to the model and receive fraud predictions in real time.  |

## Workflow

### Data Preparation

The first step in the pipeline focuses on preparing transaction data so it can be analyzed effectively.

We start by standardizing the transaction amounts so they are on the same scale as the other data. This helps the model compare information more fairly.

Next, we separate the data into two parts:

* the information the model uses to make predictions, and
* the label that tells us whether a transaction is fraud or normal.

Finally, we remove the transaction time field, since it doesn’t help identify fraud in this case.

These steps make the data consistent and ready for the model to learn from.

In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

### Model Training

We train the model using Isolation Forest, a method designed to find unusual patterns in large datasets.

Instead of learning what fraud looks like, the model focuses on finding transactions that stand out from the rest. It does this by repeatedly splitting the data and seeing which transactions are easier to separate from normal ones. Transactions that are isolated quickly are more likely to be unusual or potentially fraudulent.

Isolation Forest is commonly used for fraud detection and security monitoring because it works well with large datasets and runs quickly.

In this example, we also set the model to expect about the same proportion of fraud cases that exist in the dataset, which helps it identify anomalies more realistically.

In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### Model Evaluation

To understand how well the model works, we look at several performance measures that tell us how accurately it identifies normal transactions and fraud.

The model performs extremely well at recognizing normal transactions. In our data, it correctly identifies almost all legitimate purchases, which means it rarely flags a normal transaction as fraud.

However, detecting fraud is much harder. When the model flags a transaction as fraud, it is correct about 29% of the time. It also only identifies about 28% of all actual fraud cases, meaning most fraudulent transactions are still missed.

Although the model shows an overall accuracy of 99.9%, this number can be misleading. Fraud is very rare compared to normal transactions, so a model can appear highly accurate simply by labeling almost everything as legitimate.

Because of this, we focus on measures that better reflect fraud detection performance, which show that the model is strong at confirming normal activity but still needs improvement to reliably detect fraud.

In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

### Model Registration

The model is stored in Azure ML for reuse and governance.

In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


### Visualizing Predicted Fraud

The chart shows how many transactions the model predicts as normal versus fraud.

* 0 means the model believes the transaction is normal.
* 1 means the model believes the transaction is fraud or unusual.

In most cases, you will see a very large number of normal transactions and only a small number of flagged ones. This is expected because fraud is very rare in the dataset.

If the number of transactions flagged as fraud is roughly similar to the real number of fraud cases, it suggests the model is reasonably balanced.

This chart provides a quick check of the model’s behavior:

* Flagging too many transactions may cause unnecessary alerts.
* Flagging too few may mean fraud is being missed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


### Visualizing Transaction Amounts

The chart compares transaction amounts for transactions the model predicts as normal versus fraud.

* 0 = normal transaction
* 1 = suspected fraud

Each box shows how transaction amounts are spread within each group. The line in the middle represents the typical (median) transaction amount, while dots outside the box are unusually high or low values.

If transactions predicted as fraud have more extreme or widely spread amounts, it suggests the model is flagging unusual transaction sizes as suspicious.

This visualization helps us see what patterns the model considers suspicious and whether it may be focusing too much on unusually large or small transactions.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


### Understanding Which Factors Influence Fraud Detection

The chart below uses SHAP, a tool that helps explain why the model makes certain decisions.

Each dot represents a transaction, and each row represents a factor in the data (such as transaction characteristics). The color shows whether the value is high or low, while the position shows how much that factor pushed the model toward predicting fraud or a normal transaction.

Factors shown near the top of the chart are the most influential in the model’s decisions. For example, if a factor consistently pushes predictions toward fraud, it suggests that certain patterns in that variable are associated with suspicious transactions.

We only analyze the first 100 transactions here to keep the visualization fast and easy to read.

Overall, this analysis helps us see which factors the model relies on most, making the system more transparent and helping guide future improvements.

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)

### Cost-Benefit Analysis: False Positives vs Missed Fraud

When managing fraud, we must balance between customer experience and financial security, and decide where to set the sensitivity of the fraud detection system.

* The Cost of Being Too Strict (False Positives)
    * Impact - Customers face declined cards and frustrating support calls. Over time, this erodes brand loyalty and drives users to competitors.
    * Cost - Primarily operational (support overhead) and some reputational damage. 

* The Cost of Being Too Lenient (Missed Fraud)
    * Impact - We face immediate theft of funds, potential regulatory fines, and the risk of becoming an easy target for organized fraud rings.
    * Cost - Direct financial loss and a hit to our standing with regulators and partners

While both outcomes are undesirable, the financial and legal consequence of missing fraud are generally much higher than the cost of a false positive. For this reason, higher sensitivity is recommended — we would rather occasionally inconvenience a customer to ensure we are stopping the high-value threats.


### Recommendations for Model Improvement

To improve performance and operational value, the following enhancements are recommended.

* Additional Features - We need to introduce additional data points, such as transaction frequency and merchant risk scores, to help the model detect more fraud patterns.
* Alternate Algorithms - Consider testing additional algorithms (XGBoost or LightGBM) to assess if the resulting trained models are more effective at fraud detection.
* Retrain Periodically - Fraud patterns can change over time, and this necessitates periodic retraining of the model with updated datasets.

### Recommendations for Deployment

The system should follow a real-time flow without disruption customer transactions.

1. A customer initiates a transaction
2. Fraud detection system invokes the ML endpoint to pass the transaction data to the model
3. The model calculates and returns a risk score for the specific transaction
4. Fraud detection system makes a decision based on the risk score
    * Low Risk - the transaction is approved instantly
    * High Risk - the transaction is flagged and escalated to the Fraud Investigation Team for manual review

### Risk Assessment and Mitigation

| Risk                                          | Mitigation                                                  |
| --------------------------------------------- | ----------------------------------------------------------- |
| Fraud patterns change over time               | Retrain model periodically with updated dataset             |
| Model bios - over-flag certain demographic    | Perform fairness audits and ensure fair data representation |
| False positives overwhelming support teams    | Adjust detection thresholds                                 |

### Stakeholder Communication Plan - Model Limitations

1. Key Messages
    * Not a perfect fraud detector: The model flags unusual transactions, but not all flagged transactions are fraud, and some fraud may be missed.
    * Alerts need human review: Operations teams must verify flagged transactions before action.
    * Performance can change over time: Customer behavior and fraud tactics evolve, so the model needs ongoing monitoring and periodic updates.
    * Limited explanation: The model provides an anomaly score but cannot fully explain why it flagged a transaction.

2. Communication Approach

| Audience          | What to Share                                           | Frequency           |
| ----------------- | ------------------------------------------------------- | ------------------- |
| Executives        | High-level performance, alert trends, risks             | Monthly             |
| Risk & Compliance | False positive/negative trends, regulatory implications | Quarterly           |
| Operations        | Guidance for handling flagged transactions              | Ongoing             |